# 03. Pesquisa informada: Greedy, A\* e A\* ponderado

Este notebook trata a pesquisa informada do projeto (`src/informed.py`). Formula o problema como pesquisa, apresenta as quatro heurísticas e compara o Greedy, o A\* e
o A\* ponderado em custo, nós expandidos e tempo, com duas formulações diferentes das ações.

## O problema

Uma plataforma liga pequenos produtores do distrito de Braga aos clientes através de um centro de
recolha com dois veículos. Cada produto é recolhido no produtor, descarregado no armazém e, mais tarde,
carregado e entregue ao cliente. O agente tem de servir todas as encomendas com o menor número de km,
respeitando as janelas de recolha e de entrega (que limitam a hora a que o serviço começa), a
capacidade dos veículos, 10 min de serviço por paragem, 15 min para descarregar e 15 para carregar no
armazém, e no máximo 24 h de permanência lá.

O modelo (`src/model.py`) junta as linhas de encomenda em pedidos. Um pedido de recolha (P1, P2, ...)
agrupa as linhas de um produtor cujas janelas se intersetam; um pedido de entrega (D1, D2, ...) é uma
encomenda e depende das recolhas que trazem os seus produtos. Cada veículo faz uma sequência de viagens
que saem do centro e voltam lá, cada uma só com recolhas ou só com entregas, e nenhuma passa a
meia-noite. Os veículos saem *just in time*, para chegar à primeira paragem quando a janela abre, e um
veículo pode entregar o que outro recolheu, desde que já esteja descarregado. O tiny tem 18 pedidos e
o small 37.

`evaluate(problem, solution)` simula uma solução e devolve os km e as violações; `check_solution` volta
a verificá-la linha a linha, diretamente sobre os CSV.

In [ ]:
import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.model import load_problem, evaluate, check_solution, State, plot_routes, plot_gantt
from src.informed import (greedy, astar, astar_trips, weighted_astar, catalog, successors, HEURISTICS,
                          h_mst, h_relaxed)

RUN_HEAVY = False  # True volta a correr as pesquisas longas em vez de ler reports/

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 170)
ROOT = Path.cwd().parent
REPORTS = ROOT / "reports"
tiny = load_problem(ROOT / "data" / "tiny")
small = load_problem(ROOT / "data" / "small")
PROBLEMS = {"tiny": tiny, "small": small}


def load_solution(filename):
    """Lê uma solução gravada em reports/ como {veículo: [viagens]}."""
    return {int(vehicle): trips for vehicle, trips in json.loads((REPORTS / filename).read_text()).items()}


def print_routes(solution):
    for vehicle, trips in solution.items():
        print(f"Veículo {vehicle}:", " | ".join("W→" + "→".join(trip) + "→W" for trip in trips))


heuristics = {name: heuristic for name, heuristic in HEURISTICS.items() if name != "h0_zero"}

In [ ]:
# os 18 pedidos do tiny depois do agrupamento
rows = [dict(id=request_id, tipo=request.kind, local=request.loc,
             janela=f"{tiny.clock(request.tw[0])} – {tiny.clock(request.tw[1])}",
             kg=request.weight, linhas=len(request.items), depende_de=", ".join(request.deps))
        for request_id, request in tiny.requests.items()]
pd.DataFrame(rows).set_index("id").T

## 1. Formulação

Um estado é uma solução parcial. Para cada veículo guarda onde está, a hora, a viagem em curso (tipo e
pedidos) e as viagens já fechadas; guarda também os pedidos já servidos e a hora a que cada recolha
chegou ao armazém. No estado inicial, os dois veículos estão no centro às 00:00 do dia 30 e nada foi
servido. O objetivo é ter tudo servido e os veículos de volta ao centro. O custo g de um caminho são os
km já percorridos.

Cada ação mexe num só veículo: ou visita um pedido por servir compatível com a viagem em curso (ou abre
uma viagem nova), ou fecha a viagem e volta ao centro. Uma entrega só pode ser visitada quando todas as
suas recolhas já estão no armazém. Os sucessores que falham uma janela, a capacidade, a permanência ou
a regra da meia-noite são descartados.

Há duas podas, ambas exatas: nunca cortam uma solução viável. A primeira deteta becos sem saída, ou
seja, um pedido que, mesmo no cenário mais otimista, já não consegue começar dentro da janela. Nas
entregas, esse cenário conta com a hora mais cedo a que as recolhas podem estar no armazém, e o limite
de 24 h é testado com a hora de carga mais cedo possível da viagem, que pode vir de outra entrega com
janela anterior. Uma primeira versão usava a hora de carga *just in time* da própria entrega e, num caso
de teste, cortava a única solução viável; ficou um teste unitário para isso. A segunda poda é por
dominância: entre dois estados com os mesmos pedidos servidos e as mesmas viagens em curso, descarta-se
o que tem mais custo e relógios mais tardios. As horas de chegada ao armazém só são comparadas quando o
limite de 24 h não pode ser violado; nos outros casos têm de ser iguais.

### Segunda formulação: uma ação, uma viagem

Na formulação anterior, a mesma solução pode ser construída por muitas sequências de ações diferentes.
Na segunda, todos os veículos estão no centro entre ações, e cada ação é "o veículo v faz a viagem T".

Antes da pesquisa constrói-se um catálogo com todas as viagens que podem entrar numa solução (conjunto
de pedidos e ordem de visita), usando a partida mais cedo de cada grupo, já que partir mais tarde nunca
torna viável uma viagem inviável. Para uma ordem fixa, o horário tem forma fechada: o regresso ao centro
é `max(D + δ, c)`, em que D é a hora de partida, δ a duração sem esperas e c o regresso mais cedo que as
janelas permitem, e a viagem é viável se `D ≤ Dmax`. Entre duas ordens do mesmo conjunto, descarta-se a
que é pior ou igual em km, δ, c, Dmax e hora de carga, porque perde para qualquer partida e qualquer
continuação.

Na pesquisa, cada viagem é testada no contexto real (veículo livre, hora a que as recolhas chegaram de
facto ao armazém, capacidade do veículo, 24 h, meia-noite) com as regras do simulador, e um teste
unitário confirma que os dois horários coincidem. O estado, o custo, as podas e as heurísticas são os
da primeira formulação. Cada passo é maior: um nó expandido corresponde a uma viagem acrescentada.

## 2. Heurísticas e algoritmos

As quatro heurísticas são admissíveis: dão um limite inferior dos km que faltam.

h1 (desvio máximo) é o máximo, sobre os pedidos por servir, da distância do veículo mais próximo até ao
pedido mais a distância do pedido ao centro. Pela desigualdade triangular, nenhum completamento é mais
curto. É simples e fraca.

h2 (arcos mínimos) agrupa os pedidos por nó (local, dia, tipo), sabendo que só nós do mesmo dia e tipo
podem partilhar viagem. Cada nó por servir precisa de um arco de entrada (do centro, de outro nó do
grupo ou de um veículo com uma viagem desse grupo em curso) e cada viagem que ainda falta precisa de um
arco de regresso. A heurística soma os mínimos.

h3 (MST por grupo) parte de uma observação: se tirarmos o arco de regresso a cada viagem que falta, o
que sobra liga o centro a todos os nós por servir e custa pelo menos uma árvore de cobertura mínima.
Junta-lhe um limite do tipo *1-tree* e o número mínimo de viagens de cada grupo, calculado a partir do
peso e dos pares de pedidos que não podem ir juntos (janelas que não encadeiam, ou entregas cujos
produtos ficam prontos tarde demais para a outra).

h4 (relaxação exata por grupo) resolve de forma exata, em cada grupo (dia, tipo), um problema mais
simples: veículos ilimitados com a capacidade do maior, e entregas que saem assim que os produtos podem
estar no armazém. Mantém as janelas, os tempos de serviço e de carga e descarga, a capacidade e a regra
da meia-noite. Enumera as viagens possíveis do grupo e escolhe, por programação dinâmica sobre
subconjuntos, a combinação mais barata que cobre os pedidos por servir. Como o problema real tem mais
restrições, não pode custar menos. Por cada viagem em curso subtrai d(centro, veículo), e em cada grupo
fica com o máximo entre este valor e a h3.

O Greedy usa f = h e expande sempre o veículo com o relógio mais atrasado, para não deixar um veículo
saltar para o dia seguinte enquanto o outro ainda tem trabalho. O A\* usa f = g + h sem essa regra, e
por isso é ótimo neste espaço. Testámos também o A\* ponderado, com f = g + w·h.

## 3. Valor das heurísticas

Quanto mais perto do custo real estiver a heurística, menos nós o A\* expande. A tabela compara o valor
de cada uma no estado inicial com o melhor custo conhecido.

In [ ]:
best = json.loads((REPORTS / "best.json").read_text())
rows = []
for name, problem in PROBLEMS.items():
    initial = State.initial(problem)
    for heuristic_name, heuristic in heuristics.items():
        value = heuristic(problem, initial)
        rows.append(dict(dataset=name, heuristica=heuristic_name, h0=round(value, 2),
                         melhor_conhecido=best[name]["km"], razao=round(value / best[name]["km"], 3)))
pd.DataFrame(rows).pivot(index="heuristica", columns="dataset", values=["h0", "razao"])

Para ver o comportamento ao longo da pesquisa, reconstruímos ação a ação a sequência de estados da
solução ótima do tiny e comparamos, em cada estado, h com o custo que realmente falta (C\* − g). Uma
heurística admissível fica sempre abaixo da diagonal.

In [ ]:
optimal_tiny = load_solution("solution_tiny_astar.json")
optimal_small = load_solution("solution_small_astar.json")
optimal_km = evaluate(tiny, optimal_tiny).km


def replay(problem, solution):
    """Segue a solução pela ordem cronológica das ações e devolve os estados intermédios."""
    events = []
    for trip in evaluate(problem, solution).trips:
        for request_id, _, start in trip.stops:
            events.append((start, trip.vid, ("visit", request_id)))
        events.append((trip.back, trip.vid, ("close", None)))
    state, states = State.initial(problem), []
    for _, vehicle_id, (action, request_id) in sorted(events):
        states.append(state)
        i = [vehicle.vid for vehicle in state.vehicles].index(vehicle_id)
        others = state.vehicles[:i] + state.vehicles[i + 1:]
        for child in successors(problem, state):
            vehicle = child.vehicles[i]
            if child.vehicles[:i] + child.vehicles[i + 1:] != others:
                continue
            if (action == "visit" and vehicle.trip and vehicle.trip[-1] == request_id
                    and request_id in child.served and request_id not in state.served):
                state = child
                break
            if action == "close" and vehicle.kind is None and state.vehicles[i].kind is not None:
                state = child
                break
        else:
            raise RuntimeError(f"ação {action} {request_id} não encontrada")
    states.append(state)
    return states


path = replay(tiny, optimal_tiny)
print("estados no caminho:", len(path), "| custo final:", round(path[-1].g, 2), "| objetivo:", path[-1].is_goal(tiny))
fig, ax = plt.subplots(figsize=(6, 6))
remaining = [optimal_km - state.g for state in path]
for name, heuristic in heuristics.items():
    ax.scatter(remaining, [heuristic(tiny, state) for state in path], s=14, label=name)
ax.plot([0, optimal_km], [0, optimal_km], "k--", lw=1, label="h = custo real")
ax.set_xlabel("km que faltam (C* − g)")
ax.set_ylabel("h(estado)")
ax.set_title("tiny: admissibilidade ao longo da solução ótima")
ax.legend();
assert all(heuristic(tiny, state) <= optimal_km - state.g + 1e-6 for state in path for heuristic in heuristics.values())

## 4. Greedy best-first

In [ ]:
rows = []
for name, problem in PROBLEMS.items():
    for heuristic_name, heuristic in heuristics.items():
        for chrono in (True, False):
            result = greedy(problem, heuristic, chrono=chrono, max_expansions=30_000)
            stats = {key: value for key, value in result.summary().items() if key not in ("algorithm", "heuristic")}
            rows.append(dict(dataset=name, heuristica=heuristic_name, regra_cronologica=chrono, **stats))
pd.DataFrame(rows)

A regra cronológica expande sempre o veículo com o relógio mais atrasado. Sem ela, o Greedy pode mandar
um veículo para o dia seguinte enquanto o outro ainda tem recolhas por fazer, e depois perde-se num beco
sem saída enorme: no small não encontra solução dentro do orçamento. O A\* não usa esta regra, porque
podia cortar a solução ótima.

## 5. A\* e A\* ponderado

O A\* exato demora alguns minutos por heurística no tiny, por isso os números vêm de
`reports/informed.csv`, gerado por `run_experiments.py`. Com `RUN_HEAVY = True` voltam a ser calculados
aqui.

In [ ]:
if RUN_HEAVY:
    for name, heuristic in heuristics.items():
        print(name, astar(tiny, heuristic, max_expansions=1_000_000).summary())
informed = pd.read_csv(REPORTS / "informed.csv")
informed[["dataset", "algorithm", "heuristic_name", "cost_km", "expanded", "generated", "pruned", "seconds",
          "complete", "item_check"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, name in zip(axes, ("tiny", "small")):
    runs = informed[informed.dataset == name]
    labels = runs.algorithm + " " + runs.heuristic_name.str.split("_").str[0]
    ax.barh(labels, runs.expanded, color=["tab:green" if found else "tab:red" for found in runs.found])
    ax.set_xscale("log")
    ax.invert_yaxis()
    ax.set_title(f"{name}: nós expandidos (verde = encontrou solução)")
fig.tight_layout()

O A\* ponderado usa f = g + w·h. Quanto maior o w, mais a pesquisa confia na heurística: expande menos
nós, mas a solução pode ficar até w vezes acima do ótimo. A curva foi calculada aqui, no tiny, com h3.

In [ ]:
rows = []
for weight in (1.3, 1.4, 1.5, 1.75, 2.0, 2.5, 3.0):
    result = weighted_astar(tiny, h_mst, weight, max_expansions=100_000)
    rows.append(dict(w=weight, km=result.cost, expandidos=result.expanded, segundos=round(result.seconds, 3)))
weighted = pd.DataFrame(rows)
print(weighted)
fig, ax_km = plt.subplots(figsize=(7, 4))
ax_km.plot(weighted.w, weighted.km, "o-", color="tab:blue")
ax_km.axhline(optimal_km, color="grey", ls=":")
ax_km.set_ylabel("km", color="tab:blue")
ax_nodes = ax_km.twinx()
ax_nodes.plot(weighted.w, weighted.expandidos, "s--", color="tab:orange")
ax_nodes.set_yscale("log")
ax_nodes.set_ylabel("nós expandidos", color="tab:orange")
ax_km.set_xlabel("w")
ax_km.set_title("tiny: A* ponderado (h3)");

## 6. A\* por viagens

A segunda formulação (secção 1) escolhe uma viagem completa por ação, a partir de um catálogo
construído antes da pesquisa. A tabela mostra o tamanho do catálogo e o resultado com h3 e h4.

In [ ]:
rows = []
for name, problem in PROBLEMS.items():
    start = time.perf_counter()
    trip_catalog = catalog(problem)
    catalog_seconds = time.perf_counter() - start
    for heuristic_name in ("h3_mst", "h4_relaxed"):
        result = astar_trips(problem, heuristics[heuristic_name], max_expansions=50_000, time_limit=300)
        stats = {key: value for key, value in result.summary().items() if key not in ("algorithm", "heuristic")}
        rows.append(dict(dataset=name, heuristica=heuristic_name, ordens=trip_catalog.raw_orders,
                         opcoes=len(trip_catalog.options), catalogo_s=round(catalog_seconds, 1), **stats))
pd.DataFrame(rows)

In [ ]:
# as duas formulações lado a lado
by_stop = informed[informed.algorithm == "A*"].assign(formulacao="paragem a paragem")
by_trip = informed[informed.algorithm == "A*-viagens"].assign(formulacao="viagem a viagem")
pd.concat([by_stop, by_trip]).pivot_table(index=["dataset", "heuristic_name"], columns="formulacao",
                                          values=["expanded", "cost_km", "seconds"], aggfunc="first")

A formulação por viagens corta duas fontes de redundância. Deixa de haver ordens diferentes para
construir a mesma viagem, porque o catálogo só guarda as ordens não dominadas, e as decisões dentro de
uma viagem são tomadas todas de uma vez. Com a h4, que no estado inicial já vale o ótimo nos dois
datasets, o A\* expande 7 nós no tiny e 19 no small.

## 7. Rotas dos 2 veículos do tiny

In [ ]:
tiny_eval = evaluate(tiny, optimal_tiny)
print(tiny_eval.summary())
print("verificador:", check_solution(tiny, tiny_eval).valid)
print_routes(optimal_tiny)
plot_routes(tiny, optimal_tiny, f"tiny: A* (ótimo), {tiny_eval.km:.2f} km");
plot_gantt(tiny, tiny_eval);

In [ ]:
greedy_tiny = greedy(tiny, h_mst)
plot_routes(tiny, greedy_tiny.solution, f"tiny: Greedy h3, {greedy_tiny.cost:.2f} km");

## 8. Rotas ótimas do small

In [ ]:
small_eval = evaluate(small, optimal_small)
print(small_eval.summary())
print("verificador:", check_solution(small, small_eval).valid)
print_routes(optimal_small)
plot_routes(small, optimal_small, f"small: A* por viagens (ótimo), {small_eval.km:.2f} km");
plot_gantt(small, small_eval);

## 9. Conclusões

A h1 é fraca demais: no estado inicial vale uma pequena fração do ótimo, e o A\* com h1 esgota o
orçamento sem terminar. A h3 (MST por grupo, limite *1-tree* e número mínimo de viagens) já resolve o
tiny, mas com centenas de milhares de expansões. A h4 vale, logo no estado inicial, o próprio ótimo nos
dois datasets, e com ela e a formulação por viagens o A\* prova o ótimo do tiny e do small com 7 e 19
expansões.

O Greedy encontra soluções em menos de um segundo, mas sem garantias: no tiny fica entre 14,6% (h4) e
32,7% (h1) acima do ótimo. O A\* ponderado dá uma troca controlada entre qualidade e esforço. No tiny,
com w entre 1,4 e 1,75, encontra o ótimo com 69 a 411 expansões, contra 7 302 do A\* com h4 e 709 786
com h3.

No small (37 pedidos), o A\* paragem a paragem não termina dentro do orçamento com nenhuma heurística, e o
A\* por viagens com h4 termina. A escolha das ações conta tanto como a da heurística.